In [ ]:
///- --package .

#### combat_text (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< sm'_operators

:= digit_at (text : string) (at : i32) : option i32 =
    ? at >= sm'::length text ?> None |: convert::try_convert (sm'::substring at 1i32 text)

:~ ^ digits_from (text : string) (at : i32) (value : i32) : i32 * i32 =
    ?= digit_at text at ?|
    | Some d => digits_from text (at + 1i32) (value * 10i32 + d)
    | None => value, at

:= rest_from (at : i32) (text : string) : string = sm'::substring at (sm'::length text - at) text

:= starts_with_any (prefixes : list string) (text : string) : bool =
    listm::fold (\ found prefix => found || sm'::starts_with prefix text) !- prefixes

:== number_pattern = { before : string; ends : list string }

:= number_at (p : number_pattern) (found : i32) (text : string) : option i32 =
    := start = found + sm'::length p::before
    := value, stop = digits_from text start 0i32
    ? stop > start && starts_with_any p::ends (rest_from stop text) ?> Some value |: None

:~ ^ number_between (p : number_pattern) (text : string) : option i32 =
    := at = sm'::index_of p::before text
    ? at < 0i32
    ?> None
    |:
        ?= number_at p at text ?|
        | Some n => Some n
        | None => number_between p (rest_from (at + 1i32) text)

:= hit_amount (text : string) : option i32 =
    number_between { before = "loses "; ends = [ " hitpoints due to your"; " hitpoint due to your" ] } text

:= damage_taken (text : string) : option i32 =
    number_between { before = "You lose "; ends = [ " hitpoints due to an attack"; " hitpoint due to an attack" ] } text

:| combat_line =
    | Took : i32
    | Dealt : i32
    | Other

:= read_line (text : string) : combat_line =
    ?= damage_taken text ?|
    | Some lost => Took lost
    | None =>
        ?= hit_amount text ?|
        | Some amount => Dealt amount
        | None => Other

#### hits_read_the_amount_the_monster_loses (test)

In [1]:
///- --test --symbol
///> lua

:= crit = "A rat loses 1 hitpoint due to your critical attack."
(hit_amount "A troll loses 12 hitpoints due to your attack.", hit_amount crit) |> _assert_eq (Some 12i32, Some 1i32)

__assert_eq { expected = (Some(12), Some(1)) }

#### hits_skip_other_attackers_and_own_losses (test)

In [2]:
///- --test --symbol
///> lua

:= theirs = "A troll loses 12 hitpoints due to an attack by a cyclops."
:= mine = "You lose 30 hitpoints due to an attack by a troll."
(hit_amount theirs, hit_amount mine) |> _assert_eq (None, None)

__assert_eq { expected = (None, None) }

#### damage_reads_the_amount_the_knight_loses (test)

In [3]:
///- --test --symbol
///> lua

:= rat = "You lose 1 hitpoint due to an attack by a rat."
(damage_taken "You lose 30 hitpoints due to an attack by a troll.", damage_taken rat)
|> _assert_eq (Some 30i32, Some 1i32)

__assert_eq { expected = (Some(30), Some(1)) }

#### damage_skips_monster_losses_and_mana (test)

In [4]:
///- --test --symbol
///> lua

(damage_taken "A troll loses 12 hitpoints due to your attack.", damage_taken "You lose 5 mana.")
|> _assert_eq (None, None)

__assert_eq { expected = (None, None) }

#### a_later_match_counts_when_an_earlier_one_does_not_fit (test)

In [5]:
///- --test --symbol
///> lua

:= p = { before = "loses "; ends = [ " hitpoints" ] }
number_between p "it loses its way and loses 7 hitpoints" |> _assert_eq (Some 7i32)

__assert_eq { expected = Some(7) }

#### a_line_with_both_reads_as_damage_taken (test)

In [6]:
///- --test --symbol
///> lua

read_line "You lose 3 hitpoints due to an attack; a rat loses 4 hitpoints due to your hit" |> _assert_eq (Took 3i32)

__assert_eq { expected = Took(3) }

#### hits_and_other_lines_read_apart (test)

In [7]:
///- --test --symbol
///> lua

(read_line "a rat loses 4 hitpoints due to your hit", read_line "You lose 5 mana.")
|> _assert_eq (Dealt 4i32, Other)

__assert_eq { expected = (Dealt(4), Other) }